# Dorna2 — resumable ParsBench ParsiNLU Multiple-Choice evaluation

This notebook evaluates `PartAI/Dorna2-Llama3.1-8B-Instruct` on the ParsiNLU Multiple Choice task used by ParsBench.

It uses a fixed zero-shot Persian prompt, deterministic generation, category-level results, model-independent MCQ normalization, and Google Drive checkpointing after every example so a Colab disconnect can resume safely.

In [ ]:
# Colab setup
!pip -q install -U "parsbench>=0.2,<0.3" transformers accelerate bitsandbytes sentencepiece datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE_DIR = Path('/content/drive/MyDrive/llm_benchmark/parsbench')
BASE_DIR.mkdir(parents=True, exist_ok=True)
print('Results directory:', BASE_DIR)

In [ ]:
# Optional Hugging Face login if your runtime requires it.
# from huggingface_hub import login
# login()

## Load Dorna2 on a T4 (4-bit NF4)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'PartAI/Dorna2-Llama3.1-8B-Instruct'
MODEL_SLUG = 'dorna2_llama3.1_8b_instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
)
model.eval()
print('Loaded:', MODEL_NAME)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Load the fixed ParsiNLU MCQ benchmark

ParsBench uses ParsiNLU Multiple Choice. The original JSONL retains the `category` field, which lets us report `math_and_logic`, `common_knowledge`, and `literature` separately.

In [ ]:
from datasets import load_dataset
import pandas as pd

# For final thesis reproducibility, replace `master` with a specific Git commit SHA.
DATASET_URL = ('https://raw.githubusercontent.com/persiannlp/parsinlu/'
               'master/data/multiple-choice/train.jsonl')

ds = load_dataset('json', data_files=DATASET_URL, split='train')
eval_df = ds.to_pandas().reset_index(drop=True)
eval_df['example_id'] = range(1, len(eval_df) + 1)
print('Total examples:', len(eval_df))
print(eval_df['category'].value_counts())
eval_df.head()

## ParsBench-compatible zero-shot Persian prompt

In [ ]:
PARSBENCH_MC_QA_INSTRUCTION_FA = (
    'در ادامه، به شما یک سوال چند گزینه‌ای به زبان فارسی نشان داده می شود. '
    'شما باید بر اساس دانش خود به سوال پاسخ دهید. '
    'پاسخ خود را از بین گزینه‌های داده شده انتخاب کنید.\n'
    'فقط عدد متناظر با گزینه درست را خروجی بده.'
)

def build_parsbench_mcq_prompt(row):
    options = '\n'.join(
        f'{i}. {candidate}' for i, candidate in enumerate(row['candidates'], start=1)
    )
    return (
        f"{PARSBENCH_MC_QA_INSTRUCTION_FA}\n\n"
        f"سوال:\n'''{row['question']}'''\n"
        f"گزینه ها:\n'''{options}'''\n"
        'جواب:'
    )

print(build_parsbench_mcq_prompt(eval_df.iloc[0]))

## Deterministic generation

The ParsBench prompt is sent as the user message without adding a second task-specific system instruction. This keeps the benchmark instructions consistent across models.

In [ ]:
@torch.inference_mode()
def generate_completion(prompt, max_new_tokens=32):
    messages = [{'role': 'user', 'content': prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        formatted_prompt, return_tensors='pt', truncation=True, max_length=2048
    ).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    return generated.strip()

## MCQ normalization

The raw completion is always saved. Scoring also converts Persian/Arabic digits to ASCII and extracts a single clearly identified option from verbose responses.

In [ ]:
import re

DIGIT_TRANSLATION = str.maketrans({
    '۰':'0','۱':'1','۲':'2','۳':'3','۴':'4','۵':'5','۶':'6','۷':'7','۸':'8','۹':'9',
    '٠':'0','١':'1','٢':'2','٣':'3','٤':'4','٥':'5','٦':'6','٧':'7','٨':'8','٩':'9',
})

def normalize_mcq_answer(text):
    if text is None:
        return '-1'
    text = str(text).translate(DIGIT_TRANSLATION).strip()
    if text in {'1','2','3','4','5'}:
        return text

    pattern = r'(?:گزینه|جواب|پاسخ(?:\s+صحیح)?|answer|option)\s*(?:صحیح)?\s*[:：-]?\s*\**\s*([1-5])\b'
    m = re.search(pattern, text, flags=re.IGNORECASE)
    if m:
        return m.group(1)

    matches = re.findall(r'(?<!\d)([1-5])(?!\d)', text)
    unique = list(dict.fromkeys(matches))
    return unique[0] if len(unique) == 1 else '-1'

for sample in ['2', '۲', 'گزینه 4: A-B', 'جواب صحیح: **2**', 'جواب: 4. باران است با برف']:
    print(repr(sample), '->', normalize_mcq_answer(sample))

## Drive checkpoint helpers

In [ ]:
import os, json, time
import numpy as np

TASK_NAME = 'parsinlu_multiple_choice'
CHECKPOINT_PATH = BASE_DIR / f'{MODEL_SLUG}_{TASK_NAME}_checkpoint.csv'
SUMMARY_PATH = BASE_DIR / f'{MODEL_SLUG}_{TASK_NAME}_summary.csv'
CONFIG_PATH = BASE_DIR / f'{MODEL_SLUG}_{TASK_NAME}_config.json'

def load_checkpoint(path):
    if not path.exists():
        return {}
    df = pd.read_csv(path, dtype={'target': str, 'normalized_completion': str})
    return {int(row['example_id']): row.to_dict() for _, row in df.iterrows()}

def save_checkpoint_atomic(results, path):
    if not results:
        return
    out = pd.DataFrame(results.values()).sort_values('example_id')
    tmp_path = str(path) + '.tmp'
    out.to_csv(tmp_path, index=False, encoding='utf-8-sig')
    os.replace(tmp_path, path)

def save_run_config(extra=None):
    import transformers, parsbench
    config = {
        'model': MODEL_NAME,
        'task': 'ParsiNLU Multiple Choice',
        'dataset_url': DATASET_URL,
        'prompt_language': 'fa',
        'prompt_shots': 0,
        'do_sample': False,
        'max_new_tokens': 32,
        'quantization': '4bit-nf4-double-quant',
        'compute_dtype': 'float16',
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'torch_version': torch.__version__,
        'transformers_version': transformers.__version__,
        'parsbench_version': getattr(parsbench, '__version__', 'unknown'),
        'normalizer': 'digit normalization + labelled-option extraction + unique-number fallback',
    }
    if extra:
        config.update(extra)
    with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
        json.dump(config, f, ensure_ascii=False, indent=2)
    return config

## Resumable evaluation loop

Successful examples are skipped on restart; failed examples are retried. The checkpoint CSV is atomically replaced after every example.

In [ ]:
def run_resumable_mcq(dataframe, checkpoint_path=CHECKPOINT_PATH, limit_per_category=10):
    results = load_checkpoint(checkpoint_path)
    completed_ids = {
        eid for eid, row in results.items() if row.get('status') == 'success'
    }
    categories = ['math_and_logic', 'common_knowledge', 'literature']

    if limit_per_category is None:
        run_df = dataframe[dataframe['category'].isin(categories)].copy()
    else:
        run_df = (dataframe[dataframe['category'].isin(categories)]
                  .groupby('category', sort=False, group_keys=False)
                  .head(limit_per_category).copy())

    run_ids = set(run_df['example_id'].astype(int))
    print(f'Selected examples: {len(run_df)}')
    print(f'Already completed successfully: {len(completed_ids & run_ids)}')
    print(f'Remaining: {len(run_df) - len(completed_ids & run_ids)}')

    for _, row in run_df.iterrows():
        example_id = int(row['example_id'])
        if example_id in completed_ids:
            continue

        prompt = build_parsbench_mcq_prompt(row)
        target = str(row['answer']).translate(DIGIT_TRANSLATION).strip()

        try:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            start = time.perf_counter()
            raw_completion = generate_completion(prompt, max_new_tokens=32)
            latency = time.perf_counter() - start
            normalized = normalize_mcq_answer(raw_completion)
            output_tokens = len(tokenizer.encode(raw_completion, add_special_tokens=False))
            peak_vram_gb = (torch.cuda.max_memory_allocated() / (1024**3)
                            if torch.cuda.is_available() else np.nan)

            results[example_id] = {
                'example_id': example_id,
                'source_id': row.get('id', ''),
                'category': row['category'],
                'question': row['question'],
                'candidates': json.dumps(row['candidates'], ensure_ascii=False),
                'prompt': prompt,
                'target': target,
                'raw_completion': raw_completion,
                'normalized_completion': normalized,
                'correct': int(normalized == target),
                'latency_sec': latency,
                'output_tokens': output_tokens,
                'tokens_per_sec': output_tokens / latency if latency > 0 else np.nan,
                'peak_vram_gb': peak_vram_gb,
                'status': 'success',
                'error': '',
            }
            completed_ids.add(example_id)
        except Exception as exc:
            results[example_id] = {
                'example_id': example_id,
                'source_id': row.get('id', ''),
                'category': row['category'],
                'question': row['question'],
                'candidates': json.dumps(row['candidates'], ensure_ascii=False),
                'prompt': prompt,
                'target': target,
                'raw_completion': '',
                'normalized_completion': '-1',
                'correct': 0,
                'latency_sec': np.nan,
                'output_tokens': np.nan,
                'tokens_per_sec': np.nan,
                'peak_vram_gb': np.nan,
                'status': 'error',
                'error': repr(exc),
            }
            print(f'ERROR example {example_id}: {exc}')

        save_checkpoint_atomic(results, checkpoint_path)
        r = results[example_id]
        print(f"[{len(completed_ids & run_ids)}/{len(run_df)}] saved id={example_id} "
              f"category={row['category']} target={target} "
              f"pred={r['normalized_completion']} raw={r['raw_completion']!r}")

    return pd.DataFrame(results.values()).sort_values('example_id').reset_index(drop=True)

## 30-question smoke test (10 per category)

In [ ]:
save_run_config({'mode': 'smoke_test', 'limit_per_category': 10})
smoke_results = run_resumable_mcq(eval_df, CHECKPOINT_PATH, limit_per_category=10)

smoke_ids = set(
    eval_df[eval_df['category'].isin(['math_and_logic','common_knowledge','literature'])]
    .groupby('category', sort=False, group_keys=False).head(10)['example_id'].astype(int)
)
smoke_view = smoke_results[
    smoke_results['example_id'].isin(smoke_ids) & (smoke_results['status'] == 'success')
].copy()
smoke_summary = (smoke_view.groupby('category')
    .agg(n=('correct','size'), correct=('correct','sum'), accuracy=('correct','mean'),
         avg_latency_sec=('latency_sec','mean'), avg_tokens_per_sec=('tokens_per_sec','mean'),
         peak_vram_gb=('peak_vram_gb','max')).reset_index())
smoke_summary

## Full benchmark

Run this only after the smoke test looks correct. It reuses the same checkpoint, so successful smoke-test examples are skipped automatically.

In [ ]:
save_run_config({'mode': 'full', 'limit_per_category': None})
full_results = run_resumable_mcq(eval_df, CHECKPOINT_PATH, limit_per_category=None)
successful = full_results[full_results['status'] == 'success'].copy()

category_summary = (successful.groupby('category')
    .agg(n=('correct','size'), correct=('correct','sum'), accuracy=('correct','mean'),
         avg_latency_sec=('latency_sec','mean'), avg_tokens_per_sec=('tokens_per_sec','mean'),
         peak_vram_gb=('peak_vram_gb','max')).reset_index())
overall = pd.DataFrame([{
    'category':'OVERALL', 'n':len(successful), 'correct':int(successful['correct'].sum()),
    'accuracy':float(successful['correct'].mean()),
    'avg_latency_sec':float(successful['latency_sec'].mean()),
    'avg_tokens_per_sec':float(successful['tokens_per_sec'].mean()),
    'peak_vram_gb':float(successful['peak_vram_gb'].max()),
}])
summary = pd.concat([category_summary, overall], ignore_index=True)
summary.to_csv(SUMMARY_PATH, index=False, encoding='utf-8-sig')
print('Checkpoint:', CHECKPOINT_PATH)
print('Summary:', SUMMARY_PATH)
summary

## Audit formatting failures and runtime errors

In [ ]:
audit_df = full_results if 'full_results' in globals() else smoke_results
print('Invalid/unparseable outputs:')
display(audit_df[(audit_df['status']=='success') & (audit_df['normalized_completion']=='-1')][
    ['example_id','category','target','raw_completion']].head(30))
print('Runtime errors:')
display(audit_df[audit_df['status']=='error'][['example_id','category','error']].head(30))